# Bab 17: Dari Jaringan Saraf ke Deep Learning

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab17/atensi_ulasan.py": "import numpy as np\nimport torch\nfrom torch import nn\nfrom sklearn.feature_extraction.text import CountVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom data_ulasan import buat_data\n\nteks, y = buat_data()\nfor ng in [(1, 1), (1, 2)]:                                     # pembanding bag of words  #<1>\n    cv = CountVectorizer(ngram_range=ng, token_pattern=r\"\\S+\"); B = cv.fit_transform(teks)\n    lr = LogisticRegression(max_iter=3000).fit(B[:5000], y[:5000])\n    print(f\"bag of words {ng}: akurasi uji {lr.score(B[5000:], y[5000:]):.3f}\")\n\nkosa = {w: i + 2 for i, w in enumerate(sorted({w for t in teks for w in t.split()}))}\nL = 16                                                          # 0 = isian, 1 = [CLS]\ndef kode(t):\n    ids = [1] + [kosa[w] for w in t.split()]\n    return ids + [0] * (L - len(ids))\nT = torch.tensor([kode(t) for t in teks]); Y = torch.tensor(y)\n\nclass Atensi(nn.Module):                                        # satu kepala atensi  #<2>\n    def __init__(self, d):\n        super().__init__(); self.q, self.k, self.v = (nn.Linear(d, d) for _ in range(3))\n    def forward(self, h, topeng):\n        s = self.q(h) @ self.k(h).transpose(1, 2) / h.shape[-1] ** 0.5\n        a = torch.softmax(s.masked_fill(topeng[:, None, :], -1e9), -1)\n        return a @ self.v(h), a\n\nclass Transformer(nn.Module):\n    def __init__(self, V, d=32):\n        super().__init__()\n        self.emb, self.pos = nn.Embedding(V, d), nn.Embedding(L, d)   # posisi  #<3>\n        self.at1, self.at2 = Atensi(d), Atensi(d)\n        self.ff = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, d))\n        self.n1, self.n2, self.n3 = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)\n        self.keluar = nn.Linear(d, 2)\n    def forward(self, x):\n        topeng = x == 0\n        h = self.emb(x) + self.pos(torch.arange(L))\n        a, w1 = self.at1(h, topeng); h = self.n1(h + a)                  # residual  #<4>\n        a, w2 = self.at2(h, topeng); h = self.n2(h + a)\n        h = self.n3(h + self.ff(h))\n        return self.keluar(h[:, 0]), w2                                   # vektor [CLS]\n\ntorch.manual_seed(0)\nnet = Transformer(len(kosa) + 2); opt = torch.optim.Adam(net.parameters(), 2e-3)\nfor ep in range(15):\n    for idx in torch.randperm(5000, generator=torch.Generator().manual_seed(ep)).split(64):\n        opt.zero_grad(); nn.functional.cross_entropy(net(T[idx])[0], Y[idx]).backward(); opt.step()\nnet.eval()\nwith torch.no_grad():\n    akur = (net(T[5000:])[0].argmax(1) == Y[5000:]).float().mean()\n    print(f\"transformer 2 lapis atensi: akurasi uji {akur:.3f}\")\n    for t in [\"pengirimannya tidak lambat\", \"kurirnya ramah tapi harganya mahal\"]:\n        logit, w = net(torch.tensor([kode(t)]))\n        p = torch.softmax(logit, 1)[0, 1].item()\n        kata = t.split(); bobot = w[0, 0, 1:len(kata) + 1]                # atensi [CLS]  #<5>\n        print(f\"'{t}': p(positif) = {p:.2f}\")\n        print(\"   atensi:\", \" \".join(f\"{k}:{b:.2f}\" for k, b in zip(kata, bobot)))\n",
"bab17/cnn_daun.py": "import torch\nfrom torch import nn\nfrom data_daun import buat_data, KELAS\n\nX, y = buat_data()\nX, y = torch.tensor(X), torch.tensor(y)\nXl, yl, Xu, yu = X[:800], y[:800], X[3000:], y[3000:]       # data latih sedikit\n\ndef mlp():\n    return nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 128), nn.ReLU(),\n                         nn.Linear(128, 4))\n\ndef cnn():                                                   #<1>\n    return nn.Sequential(\n        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16 x 16 x 16\n        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32 x 8 x 8\n        nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),\n        nn.AdaptiveMaxPool2d(1), nn.Flatten(),                        # maks. global  #<2>\n        nn.Linear(32, 4))\n\ndef latih(buat, epoch=40):\n    torch.manual_seed(0); net = buat()\n    opt = torch.optim.Adam(net.parameters(), 2e-3); f = nn.CrossEntropyLoss()\n    for ep in range(epoch):\n        for idx in torch.randperm(800, generator=torch.Generator().manual_seed(ep)).split(32):\n            opt.zero_grad(); f(net(Xl[idx]), yl[idx]).backward(); opt.step()\n    return net\n\nfor nama, buat in [(\"MLP\", mlp), (\"CNN\", cnn)]:\n    net = latih(buat).eval()\n    with torch.no_grad():\n        p = net(Xu).argmax(1)\n        geser = net(torch.roll(Xu, shifts=(8, 8), dims=(2, 3))).argmax(1)   # citra digeser  #<3>\n    n_par = sum(w.numel() for w in net.parameters())\n    print(f\"{nama}: {n_par:6d} parameter, akurasi uji {(p == yu).float().mean():.3f},\"\n          f\" uji digeser {(geser == yu).float().mean():.3f}\")\n    per_kelas = [(p[yu == k] == k).float().mean().item() for k in range(4)]\n    print(\"     per kelas:\", \", \".join(f\"{n} {a:.2f}\" for n, a in zip(KELAS, per_kelas)))\n",
"bab17/data_daun.py": "\"\"\"Citra potongan daun padi sintetis 32 x 32 RGB untuk 4 kelas (ilustrasi).\n\n0 sehat, 1 blas (bercak belah ketupat berpusat abu-abu), 2 bercak cokelat\n(bercak bulat kecil), 3 hawar daun bakteri (menguning dari tepi). Pola dibuat\nsangat disederhanakan dan bukan pengganti citra lapangan.\n\"\"\"\nimport numpy as np\n\nKELAS = [\"sehat\", \"blas\", \"bercak cokelat\", \"hawar\"]\n\ndef citra(k, rng, S=32):\n    yy, xx = np.mgrid[0:S, 0:S] / S\n    hijau = np.array([0.25, 0.55, 0.2]) + rng.normal(0, 0.04, 3)\n    I = np.ones((S, S, 3)) * hijau\n    I *= (1 + 0.08 * np.sin(2 * np.pi * (xx * rng.uniform(5, 8) + rng.uniform())))[..., None]  # urat daun\n    if k == 1:                                        # blas: belah ketupat\n        for _ in range(rng.integers(1, 3)):\n            cx, cy = rng.uniform(0.1, 0.9, 2); a, b = rng.uniform(0.04, 0.08), rng.uniform(0.1, 0.2)\n            d = np.abs(xx - cx) / a + np.abs(yy - cy) / b\n            I[d < 1] = [0.45, 0.3, 0.15]; I[d < 0.5] = [0.7, 0.7, 0.65]\n    if k == 2:                                        # bercak cokelat: bulat kecil\n        for _ in range(rng.integers(2, 6)):\n            cx, cy = rng.uniform(0.05, 0.95, 2); r = rng.uniform(0.025, 0.05)\n            I[(xx - cx) ** 2 + (yy - cy) ** 2 < r ** 2] = [0.45, 0.25, 0.1]\n    if k == 3:                                        # hawar: menguning dari satu tepi\n        sisi = rng.integers(2); lebar = rng.uniform(0.08, 0.3)\n        tepi = (xx if sisi == 0 else 1 - xx) + 0.05 * np.sin(yy * rng.uniform(6, 12))\n        I[tepi < lebar] = [0.8, 0.75, 0.35]\n    for _ in range(rng.integers(0, 4)):               # kotoran/bayangan di semua kelas\n        cx, cy = rng.uniform(0, 1, 2); r = rng.uniform(0.02, 0.05)\n        I[(xx - cx) ** 2 + (yy - cy) ** 2 < r ** 2] *= 0.5\n    I *= rng.uniform(0.7, 1.3)                        # pencahayaan\n    I += rng.normal(0, 0.06, I.shape)                 # derau kamera\n    return np.clip(I, 0, 1).astype(np.float32)\n\ndef buat_data(n=4000, seed=0):\n    rng = np.random.default_rng(seed)\n    y = rng.integers(0, 4, n)\n    X = np.stack([citra(k, rng) for k in y]).transpose(0, 3, 1, 2)   # N, C, H, W\n    return X, y\n",
"bab17/data_ulasan.py": "\"\"\"Ulasan pendek layanan daring berbahasa Indonesia yang dibangkitkan dari pola\n(ilustrasi). Polaritas dibalik oleh negasi (\"tidak\", \"kurang\", \"nggak\"), dan\npada kalimat \"X tapi Y\" yang menentukan adalah klausa kedua.\n\"\"\"\nimport numpy as np\n\nSUBJEK = [\"drivernya\", \"pengirimannya\", \"aplikasinya\", \"barangnya\", \"pelayanannya\",\n          \"harganya\", \"kurirnya\", \"makanannya\", \"penjualnya\", \"kemasannya\"]\nPOS = [\"bagus\", \"cepat\", \"ramah\", \"murah\", \"memuaskan\", \"mantap\", \"rapi\", \"enak\"]\nNEG = [\"buruk\", \"lambat\", \"kasar\", \"mahal\", \"mengecewakan\", \"lemot\", \"berantakan\", \"hambar\"]\nKETERANGAN = [\"\", \"\", \"sangat\", \"cukup\", \"agak\", \"benar-benar\"]\nNEGASI = [\"tidak\", \"kurang\", \"nggak\"]\nPENGISI = [\"\", \"\", \"\", \"kak\", \"sih\", \"ya\", \"deh\", \"pokoknya\"]\n\ndef _klausa(rng):\n    s = rng.choice(SUBJEK); pos = rng.random() < 0.5\n    adj = rng.choice(POS if pos else NEG)\n    neg = rng.random() < 0.35\n    if neg:                                              # \"kurang\" hanya untuk kata positif\n        ng = rng.choice(NEGASI if pos else [\"tidak\", \"nggak\"])\n        kata = [s, ng, rng.choice([\"\", \"\", \"begitu\"] + ([] if ng == \"kurang\" else [\"terlalu\"])), adj]\n    else:\n        kata = [s, rng.choice(KETERANGAN), adj]\n    return [w for w in kata if w], pos != neg            # negasi membalik polaritas\n\ndef buat_data(n=6000, seed=0):\n    rng = np.random.default_rng(seed)\n    teks, y = [], []\n    for _ in range(n):\n        k1, p1 = _klausa(rng)\n        if rng.random() < 0.4:                           # \"X tapi Y\"\n            k2, p2 = _klausa(rng)\n            kata, p = k1 + [\"tapi\"] + k2, p2\n        else:\n            kata, p = k1, p1\n        isi = rng.choice(PENGISI)\n        teks.append(\" \".join(kata + ([isi] if isi else []))); y.append(int(p))\n    return teks, np.array(y)\n",
"bab17/indobert_sentimen.py": "# Fine-tuning IndoBERT untuk analisis sentimen ulasan (kerangka).\n# Memerlukan: pip install transformers datasets, dan koneksi internet untuk\n# mengunduh model (sekitar 500 MB). Skrip ini TIDAK dijalankan untuk buku ini.\nimport torch\nfrom transformers import AutoTokenizer, AutoModelForSequenceClassification\nfrom data_ulasan import buat_data\n\nnama = \"indobenchmark/indobert-base-p1\"                       # model fondasi  #<1>\ntok = AutoTokenizer.from_pretrained(nama)\nmodel = AutoModelForSequenceClassification.from_pretrained(nama, num_labels=2)  #<2>\n\nteks, y = buat_data()                                         # ganti dengan data nyata\nenc = tok(teks[:5000], padding=True, truncation=True, max_length=64, return_tensors=\"pt\")\nlabel = torch.tensor(y[:5000])\nopt = torch.optim.AdamW(model.parameters(), lr=2e-5)          # laju belajar kecil  #<3>\nmodel.train()\nfor ep in range(2):\n    for idx in torch.randperm(5000).split(32):\n        keluar = model(input_ids=enc[\"input_ids\"][idx],\n                       attention_mask=enc[\"attention_mask\"][idx], labels=label[idx])\n        opt.zero_grad(); keluar.loss.backward(); opt.step()   # loss = cross-entropy\n",
"bab17/vae_glif.py": "import sys, pathlib\nimport torch\nfrom torch import nn\nsys.path.insert(0, str(pathlib.Path(__file__).parent.parent / \"bab16\"))\nfrom data_aksara import buat_data                            # glif dari Bab 16\n\nX, y = buat_data()\nX = torch.tensor(X); Xl, Xu = X[:5000], X[5000:]\nZ = 2                                                         # dimensi laten\n\nclass VAE(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.enc = nn.Sequential(nn.Linear(400, 256), nn.ReLU(), nn.Linear(256, 2 * Z))  #<1>\n        self.dec = nn.Sequential(nn.Linear(Z, 256), nn.ReLU(), nn.Linear(256, 400))\n    def forward(self, x):\n        mu, logvar = self.enc(x).chunk(2, 1)                 # q(z | x) Gaussian\n        z = mu + (0.5 * logvar).exp() * torch.randn_like(mu) # reparameterisasi  #<2>\n        return self.dec(z), mu, logvar\n\ndef elbo(net, x):                                             #<3>\n    logit, mu, logvar = net(x)\n    rek = -nn.functional.binary_cross_entropy_with_logits(logit, x, reduction=\"none\").sum(1)\n    kl = 0.5 * (mu ** 2 + logvar.exp() - 1 - logvar).sum(1)  # KL(q || N(0, I))\n    return rek.mean(), kl.mean()\n\ntorch.manual_seed(0)\nnet = VAE(); opt = torch.optim.Adam(net.parameters(), 1e-3)\nfor ep in range(30):\n    for idx in torch.randperm(5000, generator=torch.Generator().manual_seed(ep)).split(64):\n        rek, kl = elbo(net, Xl[idx]); opt.zero_grad(); (kl - rek).backward(); opt.step()\n    if ep % 10 == 9:\n        with torch.no_grad():\n            rek, kl = elbo(net, Xu)\n        print(f\"epoch {ep + 1}: ELBO uji {(rek - kl).item():7.1f} = rekonstruksi {rek.item():7.1f}\"\n              f\" - KL {kl.item():4.1f}\")\nfrom sklearn.neighbors import KNeighborsClassifier           # seberapa informatif z?  #<4>\nwith torch.no_grad():\n    zl, zu = net.enc(Xl).chunk(2, 1)[0], net.enc(Xu).chunk(2, 1)[0]\nknn = KNeighborsClassifier(10).fit(zl, y[:5000])\nfrom sklearn.decomposition import PCA\npca = PCA(2).fit(Xl.numpy())\nknn_p = KNeighborsClassifier(10).fit(pca.transform(Xl.numpy()), y[:5000])\nprint(f\"akurasi 10-NN di ruang 2D: VAE {knn.score(zu, y[5000:]):.3f},\"\n      f\" PCA {knn_p.score(pca.transform(Xu.numpy()), y[5000:]):.3f}\")\n",
"bab16/data_aksara.py": "\"\"\"Citra tulisan tangan sintetis 20 x 20 piksel untuk 10 kelas glif (ilustrasi).\n\nSetiap glif dibangun dari goresan sederhana (lengkung, lingkaran kecil, garis\ntegak dan mendatar) yang meniru gaya goresan aksara Jawa secara skematis.\nBentuknya TIDAK akurat sebagai aksara; untuk penelitian, gunakan himpunan data\ntulisan tangan aksara Jawa yang sebenarnya.\n\"\"\"\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\ndef _busur(cx, cy, r, a0, a1, n=12):\n    t = np.linspace(np.radians(a0), np.radians(a1), n)\n    return list(zip(cx + r * np.cos(t), cy + r * np.sin(t)))\n\ndef _glif():\n    G = []\n    G.append([_busur(.3, .5, .18, 180, 360) + [(.48, .8)], [(.48, .5), (.7, .5)], _busur(.7, .5, .15, 180, 360)])\n    G.append([[(.15, .7), (.15, .35)], _busur(.32, .35, .17, 180, 360), [(.49, .35), (.49, .7)], _busur(.66, .35, .17, 180, 360) + [(.83, .7)]])\n    G.append([_busur(.5, .4, .25, 180, 360), [(.25, .4), (.25, .75), (.75, .75), (.75, .4)], _busur(.5, .55, .07, 0, 360)])\n    G.append([[(.2, .3), (.8, .3)], [(.35, .3), (.35, .8)], [(.65, .3), (.65, .8)], _busur(.5, .8, .15, 0, 180)])\n    G.append([_busur(.35, .5, .2, 90, 450), [(.55, .5), (.85, .5)], [(.85, .25), (.85, .8)]])\n    G.append([_busur(.5, .35, .25, 180, 360), [(.25, .35), (.4, .8)], [(.75, .35), (.6, .8)], [(.4, .8), (.6, .8)]])\n    G.append([[(.2, .25), (.2, .75)], [(.2, .5), (.8, .5)], [(.8, .25), (.8, .75)], _busur(.5, .3, .1, 0, 360)])\n    G.append([_busur(.3, .4, .15, 0, 360), _busur(.7, .4, .15, 0, 360), [(.15, .75), (.85, .75)]])\n    G.append([[(.15, .3), (.35, .75), (.5, .4), (.65, .75), (.85, .3)]])\n    G.append([_busur(.5, .5, .3, 200, 520), [(.5, .2), (.5, .8)]])\n    return G\n\nGLIF = _glif()\n\ndef gambar(k, rng, ukuran=20):\n    S = 64; im = Image.new(\"L\", (S, S), 0); d = ImageDraw.Draw(im)\n    a = np.radians(rng.normal(0, 12)); sk = rng.normal(1, 0.12, 2); sh = rng.normal(0, 0.2)\n    M = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]]) @ np.array([[sk[0], sh], [0, sk[1]]])\n    geser = rng.normal(0, 0.06, 2)\n    for gores in GLIF[k]:\n        P = (np.array(gores) - 0.5 + rng.normal(0, 0.015, (len(gores), 2))) @ M.T + 0.5 + geser\n        d.line([tuple(p) for p in P * S], fill=255, width=int(rng.integers(3, 6)), joint=\"curve\")\n    x = np.asarray(im.resize((ukuran, ukuran), Image.BILINEAR), float) / 255\n    return np.clip(x + rng.normal(0, 0.1, x.shape), 0, 1)\n\ndef buat_data(n=6000, seed=0):\n    rng = np.random.default_rng(seed)\n    y = rng.integers(0, 10, n)\n    X = np.stack([gambar(k, rng) for k in y]).reshape(n, -1)\n    return X.astype(np.float32), y\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab17')
print('folder kerja:', os.getcwd())

## Listing 17.1: MLP dan CNN untuk citra daun padi sintetis

Berkas: `kode/bab17/cnn_daun.py`

In [ ]:
import torch
from torch import nn
from data_daun import buat_data, KELAS

X, y = buat_data()
X, y = torch.tensor(X), torch.tensor(y)
Xl, yl, Xu, yu = X[:800], y[:800], X[3000:], y[3000:]       # data latih sedikit

def mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 128), nn.ReLU(),
                         nn.Linear(128, 4))

def cnn():                                                   # (1)
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16 x 16 x 16
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32 x 8 x 8
        nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
        nn.AdaptiveMaxPool2d(1), nn.Flatten(),                        # maks. global  # (2)
        nn.Linear(32, 4))

def latih(buat, epoch=40):
    torch.manual_seed(0); net = buat()
    opt = torch.optim.Adam(net.parameters(), 2e-3); f = nn.CrossEntropyLoss()
    for ep in range(epoch):
        for idx in torch.randperm(800, generator=torch.Generator().manual_seed(ep)).split(32):
            opt.zero_grad(); f(net(Xl[idx]), yl[idx]).backward(); opt.step()
    return net

for nama, buat in [("MLP", mlp), ("CNN", cnn)]:
    net = latih(buat).eval()
    with torch.no_grad():
        p = net(Xu).argmax(1)
        geser = net(torch.roll(Xu, shifts=(8, 8), dims=(2, 3))).argmax(1)   # citra digeser  # (3)
    n_par = sum(w.numel() for w in net.parameters())
    print(f"{nama}: {n_par:6d} parameter, akurasi uji {(p == yu).float().mean():.3f},"
          f" uji digeser {(geser == yu).float().mean():.3f}")
    per_kelas = [(p[yu == k] == k).float().mean().item() for k in range(4)]
    print("     per kelas:", ", ".join(f"{n} {a:.2f}" for n, a in zip(KELAS, per_kelas)))

## Listing 17.2: Transformer kecil dari nol untuk sentimen ulasan

Berkas: `kode/bab17/atensi_ulasan.py`

In [ ]:
import numpy as np
import torch
from torch import nn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from data_ulasan import buat_data

teks, y = buat_data()
for ng in [(1, 1), (1, 2)]:                                     # pembanding bag of words  # (1)
    cv = CountVectorizer(ngram_range=ng, token_pattern=r"\S+"); B = cv.fit_transform(teks)
    lr = LogisticRegression(max_iter=3000).fit(B[:5000], y[:5000])
    print(f"bag of words {ng}: akurasi uji {lr.score(B[5000:], y[5000:]):.3f}")

kosa = {w: i + 2 for i, w in enumerate(sorted({w for t in teks for w in t.split()}))}
L = 16                                                          # 0 = isian, 1 = [CLS]
def kode(t):
    ids = [1] + [kosa[w] for w in t.split()]
    return ids + [0] * (L - len(ids))
T = torch.tensor([kode(t) for t in teks]); Y = torch.tensor(y)

class Atensi(nn.Module):                                        # satu kepala atensi  # (2)
    def __init__(self, d):
        super().__init__(); self.q, self.k, self.v = (nn.Linear(d, d) for _ in range(3))
    def forward(self, h, topeng):
        s = self.q(h) @ self.k(h).transpose(1, 2) / h.shape[-1] ** 0.5
        a = torch.softmax(s.masked_fill(topeng[:, None, :], -1e9), -1)
        return a @ self.v(h), a

class Transformer(nn.Module):
    def __init__(self, V, d=32):
        super().__init__()
        self.emb, self.pos = nn.Embedding(V, d), nn.Embedding(L, d)   # posisi  # (3)
        self.at1, self.at2 = Atensi(d), Atensi(d)
        self.ff = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, d))
        self.n1, self.n2, self.n3 = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.keluar = nn.Linear(d, 2)
    def forward(self, x):
        topeng = x == 0
        h = self.emb(x) + self.pos(torch.arange(L))
        a, w1 = self.at1(h, topeng); h = self.n1(h + a)                  # residual  # (4)
        a, w2 = self.at2(h, topeng); h = self.n2(h + a)
        h = self.n3(h + self.ff(h))
        return self.keluar(h[:, 0]), w2                                   # vektor [CLS]

torch.manual_seed(0)
net = Transformer(len(kosa) + 2); opt = torch.optim.Adam(net.parameters(), 2e-3)
for ep in range(15):
    for idx in torch.randperm(5000, generator=torch.Generator().manual_seed(ep)).split(64):
        opt.zero_grad(); nn.functional.cross_entropy(net(T[idx])[0], Y[idx]).backward(); opt.step()
net.eval()
with torch.no_grad():
    akur = (net(T[5000:])[0].argmax(1) == Y[5000:]).float().mean()
    print(f"transformer 2 lapis atensi: akurasi uji {akur:.3f}")
    for t in ["pengirimannya tidak lambat", "kurirnya ramah tapi harganya mahal"]:
        logit, w = net(torch.tensor([kode(t)]))
        p = torch.softmax(logit, 1)[0, 1].item()
        kata = t.split(); bobot = w[0, 0, 1:len(kata) + 1]                # atensi [CLS]  # (5)
        print(f"'{t}': p(positif) = {p:.2f}")
        print("   atensi:", " ".join(f"{k}:{b:.2f}" for k, b in zip(kata, bobot)))

## Listing 17.4: VAE dengan ruang laten dua dimensi untuk glif Bab 16

Berkas: `kode/bab17/vae_glif.py`

In [ ]:
import sys, pathlib
import torch
from torch import nn
sys.path.insert(0, str(pathlib.Path.cwd().parent / "bab16"))
from data_aksara import buat_data                            # glif dari Bab 16

X, y = buat_data()
X = torch.tensor(X); Xl, Xu = X[:5000], X[5000:]
Z = 2                                                         # dimensi laten

class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(400, 256), nn.ReLU(), nn.Linear(256, 2 * Z))  # (1)
        self.dec = nn.Sequential(nn.Linear(Z, 256), nn.ReLU(), nn.Linear(256, 400))
    def forward(self, x):
        mu, logvar = self.enc(x).chunk(2, 1)                 # q(z | x) Gaussian
        z = mu + (0.5 * logvar).exp() * torch.randn_like(mu) # reparameterisasi  # (2)
        return self.dec(z), mu, logvar

def elbo(net, x):                                             # (3)
    logit, mu, logvar = net(x)
    rek = -nn.functional.binary_cross_entropy_with_logits(logit, x, reduction="none").sum(1)
    kl = 0.5 * (mu ** 2 + logvar.exp() - 1 - logvar).sum(1)  # KL(q || N(0, I))
    return rek.mean(), kl.mean()

torch.manual_seed(0)
net = VAE(); opt = torch.optim.Adam(net.parameters(), 1e-3)
for ep in range(30):
    for idx in torch.randperm(5000, generator=torch.Generator().manual_seed(ep)).split(64):
        rek, kl = elbo(net, Xl[idx]); opt.zero_grad(); (kl - rek).backward(); opt.step()
    if ep % 10 == 9:
        with torch.no_grad():
            rek, kl = elbo(net, Xu)
        print(f"epoch {ep + 1}: ELBO uji {(rek - kl).item():7.1f} = rekonstruksi {rek.item():7.1f}"
              f" - KL {kl.item():4.1f}")
from sklearn.neighbors import KNeighborsClassifier           # seberapa informatif z?  # (4)
with torch.no_grad():
    zl, zu = net.enc(Xl).chunk(2, 1)[0], net.enc(Xu).chunk(2, 1)[0]
knn = KNeighborsClassifier(10).fit(zl, y[:5000])
from sklearn.decomposition import PCA
pca = PCA(2).fit(Xl.numpy())
knn_p = KNeighborsClassifier(10).fit(pca.transform(Xl.numpy()), y[:5000])
print(f"akurasi 10-NN di ruang 2D: VAE {knn.score(zu, y[5000:]):.3f},"
      f" PCA {knn_p.score(pca.transform(Xu.numpy()), y[5000:]):.3f}")